# 02 - Simulation-based Shift Staffing Optimization

This notebook turns the earlier queueing/staffing example into a more defensible simulation-optimization model.

We choose integer staffing levels for three shifts. The objective is a **monetary total cost** that combines:

- staffing cost,
- customer waiting cost,
- a service-level penalty.

The same simulation replication seeds are used for every candidate staffing policy. This is a simple Common Random Numbers variance-reduction design.

## 1. Why the original waiting-time-only objective was incomplete

If staffing has no cost and the only objective is waiting time, adding staff is almost always beneficial. The optimizer will therefore tend to choose the maximum staffing allowed in every shift.

A useful optimization model needs an explicit trade-off. Here we place waiting and staffing in the same economic unit by assigning a monetary cost to one customer-waiting-minute.

In [ ]:
from pathlib import Path
import sys
import warnings
from itertools import product

import numpy as np
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        "Run this notebook from the repository root or from the notebooks directory."
    )

sys.path.insert(0, str((PROJECT_ROOT / "src").resolve()))

from discrete_bo import DiscreteGaussianProcessBayesOptimizer
from production_simulation import evaluate_staffing_policy

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 2. Decision space

Each shift can use between 5 and 10 staff members. The complete candidate set contains

```text
6 x 6 x 6 = 216
```

staffing combinations.

This small size lets us later verify the Bayesian Optimization result with exhaustive search. In a genuinely expensive simulation, evaluating all 216 candidates may be undesirable.

In [ ]:
staff_levels = range(5, 11)
candidates = np.array(list(product(staff_levels, repeat=3)), dtype=float)
print("Candidate count:", len(candidates))

## 3. Simulation replications

We evaluate every policy with the same replication seeds. This reduces noise when comparing policies because they face the same random demand/service scenarios.

In a real project, the number of replications should be selected using statistical precision requirements rather than chosen arbitrarily.

In [ ]:
replication_seeds = list(range(100, 112))

## 4. Economic objective

The objective is

```text
total cost
= staffing cost
+ waiting cost
+ service-level penalty
```

The service-level penalty is quadratic when the worst shift-average waiting time exceeds the target. This is a teaching formulation rather than a universal workforce-planning objective.

In [ ]:
def objective_function(x):
    staffing = np.asarray(np.rint(x), dtype=int)
    summary = evaluate_staffing_policy(
        staffing=staffing,
        replication_seeds=replication_seeds,
        hourly_staff_cost=120.0,
        waiting_minute_cost=2.0,
        target_shift_wait_minutes=1.5,
        penalty_coefficient=1500.0,
    )
    return summary.total_cost

## 5. Run discrete Bayesian Optimization

The optimizer evaluates 15 initial candidates and then 35 sequential candidates. The evaluation budget is therefore 50 out of 216 possible staffing policies.

In [ ]:
optimizer = DiscreteGaussianProcessBayesOptimizer(
    objective_function=objective_function,
    candidate_points=candidates,
    n_initial_points=15,
    xi=0.01,
    random_state=42,
)

result = optimizer.optimize(n_iterations=35, verbose=True)
best_staffing = result.best_x.astype(int)

print("Best staffing plan:", best_staffing)
print("Best estimated total cost:", round(result.best_y, 2))
print("Total simulation policy evaluations:", len(result.y_observed))

## 6. Operational summary of the selected policy

In [ ]:
summary = evaluate_staffing_policy(
    staffing=best_staffing,
    replication_seeds=replication_seeds,
    hourly_staff_cost=120.0,
    waiting_minute_cost=2.0,
    target_shift_wait_minutes=1.5,
    penalty_coefficient=1500.0,
)

print("Total cost:", round(summary.total_cost, 2))
print("Staffing cost:", round(summary.staffing_cost, 2))
print("Waiting cost:", round(summary.waiting_cost, 2))
print("Service-level penalty:", round(summary.service_level_penalty, 2))
print("Average wait (min/customer):", round(summary.average_wait_minutes, 3))
print("Average wait by shift:", np.round(summary.average_wait_by_shift, 3))
print("Average daily customer count:", round(summary.average_customer_count, 1))

## 7. Convergence

In [ ]:
best_so_far = np.minimum.accumulate(result.y_observed)

plt.figure(figsize=(9, 5))
plt.plot(np.arange(1, len(best_so_far) + 1), best_so_far, marker="o")
plt.xlabel("Number of simulation policy evaluations")
plt.ylabel("Best total cost observed so far")
plt.title("Staffing optimization convergence")
plt.grid(True)
plt.show()

## 8. Verification with exhaustive search

Because this teaching problem has only 216 candidates, we can evaluate all policies and check whether Bayesian Optimization found the same solution.

This verification would be too expensive in the class of problems for which Bayesian Optimization is normally most valuable, but it is useful for validating the educational implementation.

In [ ]:
exhaustive_results = []
for candidate in candidates:
    cost = objective_function(candidate)
    exhaustive_results.append((cost, candidate.astype(int)))

exhaustive_results.sort(key=lambda item: item[0])
exact_best_cost, exact_best_staffing = exhaustive_results[0]

print("Exhaustive-search optimum staffing:", exact_best_staffing)
print("Exhaustive-search minimum cost:", exact_best_cost)
print("Bayesian Optimization gap:", result.best_y - exact_best_cost)

## 9. Correct interpretation

The general statement is:

```text
Bayesian Optimization returns the best solution found under the evaluation budget.
```

It should not be reported as a guaranteed global optimum unless an independent exact or exhaustive method has established that fact.

## 10. Extensions for a real workforce-planning model

A realistic model may include shift start/end times, overlapping shifts, breaks, skill classes, customer priorities, absenteeism, overtime, time-varying arrival rates, and queue carryover between periods.

If shift start times become decision variables, the simulation must represent capacity on a continuous time axis; simply adding start-time variables to the current three-independent-shift model would not be sufficient.